# Lab 5 — Beat tracking: period, phase, and competing hypotheses

Tempo estimation asks **how far apart** beats are. Beat tracking must also answer **where they occur**. This lab separates those questions and illustrates why retaining multiple hypotheses can be useful.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import signal
from scipy.ndimage import gaussian_filter1d
from IPython.display import Audio, display

np.set_printoptions(precision=3, suppress=True)

def unbiased_autocorrelation(x, max_lag=None, normalize=True):
    x = np.asarray(x, dtype=float).reshape(-1)
    if max_lag is None:
        max_lag = len(x)-1
    max_lag = min(int(max_lag), len(x)-1)
    r = np.empty(max_lag+1)
    for tau in range(max_lag+1):
        a=x[:len(x)-tau]; b=x[tau:]
        r[tau] = np.dot(a,b)/max(1,len(a))
    if normalize and r[0] > 0:
        r = r/r[0]
    return r

def phase_autocorrelation(x, lags, unbiased=True):
    x=np.asarray(x,dtype=float).reshape(-1)
    lags=np.asarray(lags,dtype=int)
    P=np.zeros((len(lags), int(lags.max())))
    C=np.zeros_like(P)
    N=len(x)
    for ir,tau in enumerate(lags):
        for phi in range(tau):
            idx=np.arange(phi, N-tau, tau, dtype=int)
            if len(idx):
                P[ir,phi]=np.sum(x[idx]*x[idx+tau])
                C[ir,phi]=len(idx)
    if unbiased:
        return P/np.where(C==0,1,C), C
    return P, C

def frame_signal(x, frame_length, hop_length, center=True):
    x=np.asarray(x,float).reshape(-1)
    if center:
        p=frame_length//2
        x=np.pad(x,(p,p))
    starts=np.arange(0,max(1,len(x)-frame_length+1),hop_length,dtype=int)
    return np.stack([x[s:s+frame_length] for s in starts],axis=1), starts

def rhythmogram(x, frame_length=256, hop_length=32, lag_min=8, lag_max=96):
    frames, starts = frame_signal(x, frame_length, hop_length, center=True)
    lags=np.arange(lag_min,lag_max+1)
    R=np.zeros((len(lags),frames.shape[1]))
    w=np.hanning(frame_length)
    for j in range(frames.shape[1]):
        f=(frames[:,j]-frames[:,j].mean())*w
        ac=unbiased_autocorrelation(f, lag_max, normalize=False)
        R[:,j]=np.maximum(ac[lags],0)
        m=R[:,j].max()
        if m>0: R[:,j]/=m
    return R,lags,starts

def dp_tempo_path(R, lags, smoothness=30.0):
    R=np.asarray(R,float)
    eps=1e-10
    obs=-np.log(R+eps)
    nP,nT=R.shape
    cost=np.full((nP,nT),np.inf); back=np.zeros((nP,nT),dtype=int)
    cost[:,0]=obs[:,0]
    lp=np.log(np.asarray(lags,float))
    trans=smoothness*(lp[:,None]-lp[None,:])**2
    for t in range(1,nT):
        M=cost[:,t-1][None,:]+trans
        back[:,t]=np.argmin(M,axis=1)
        cost[:,t]=obs[:,t]+M[np.arange(nP),back[:,t]]
    path=np.zeros(nT,dtype=int); path[-1]=np.argmin(cost[:,-1])
    for t in range(nT-2,-1,-1): path[t]=back[path[t+1],t+1]
    return np.asarray(lags)[path]

def simple_novelty(x, fs, n_fft=1024, hop=256):
    f,t,Z=signal.stft(x,fs=fs,window='hann',nperseg=n_fft,noverlap=n_fft-hop,boundary=None,padded=False)
    mag=np.abs(Z)
    flux=np.maximum(mag[:,1:]-mag[:,:-1],0).sum(axis=0)
    flux=np.r_[0,flux]
    if flux.max()>0: flux/=flux.max()
    return flux,t

## 1. Same period, different phase

In [ ]:
N=300; period=24
x0=np.zeros(N); x0[4::period]=1
x1=np.zeros(N); x1[13::period]=1
r0=unbiased_autocorrelation(x0,80); r1=unbiased_autocorrelation(x1,80)
fig,ax=plt.subplots(2,1,figsize=(11,5),sharex=True)
ax[0].stem(x0,basefmt=' '); ax[0].stem(x1,linefmt='C1-',markerfmt='C1s',basefmt=' '); ax[0].set_title('Two beat sequences, same period / different phase')
ax[1].plot(r0,label='phase 4'); ax[1].plot(r1,'--',label='phase 13'); ax[1].legend(); ax[1].set_xlabel('lag'); ax[1].set_title('Autocorrelation is nearly phase-invariant'); plt.tight_layout(); plt.show()

## 2. Score candidate beat phases

Given a period estimate, evaluate several phase hypotheses by summing onset evidence at predicted beat locations.

In [ ]:
def phase_scores(odf,period):
    scores=[]
    for phi in range(period): scores.append(odf[phi::period].sum())
    return np.array(scores)
odf=gaussian_filter1d(x0 + .7*np.roll(x0,period//2),1.0)
s=phase_scores(odf,period)
plt.figure(figsize=(10,3)); plt.stem(np.arange(period),s); plt.xlabel('candidate phase'); plt.ylabel('support'); plt.title('Competing beat-phase hypotheses'); plt.show()

## 3. Keep more than one hypothesis alive

A multipath tracker does not need to commit immediately. Here we keep the three best phase hypotheses in each analysis region. A real multipath algorithm also includes transition scores and path histories; Bruno Di Giorgi's original code contains both dynamic-programming and multipath strategies.

In [ ]:
frame=120; hop=36
frames,starts=frame_signal(odf,frame,hop,center=False)
all_top=[]
for j in range(frames.shape[1]):
    sc=phase_scores(frames[:,j],period)
    top=np.argsort(sc)[-3:][::-1]
    all_top.append(top)
A=np.array(all_top)
plt.figure(figsize=(11,3));
for k in range(3): plt.plot(A[:,k],'o-',label=f'hypothesis {k+1}')
plt.xlabel('analysis frame'); plt.ylabel('phase'); plt.title('Several phase hypotheses retained through time'); plt.legend(); plt.show()

## 4. Experiment

Remove several strong onsets or insert syncopations. Notice that the strongest phase can become temporarily ambiguous. This is precisely where a one-path greedy tracker is most fragile.